## 1. Setup and Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path
import json
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')

from src.models.utils import load_config
from src.data.dataset_loader import DatasetLoader
from src.data.preprocess_audio import preprocess_audio
from src.features.extract_mfcc_features import extract_mfcc_features
from src.features.extract_spectral_features import extract_all_spectral_features
from src.features.extract_melspectrogram import extract_melspectrogram_normalized

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

config = load_config('../config.yaml')
print("Setup complete!")

## 2. Data Exploration

### 2.1 Dataset Overview

In [ ]:
dataset_loader = DatasetLoader(config['paths']['data_raw'])
file_list = dataset_loader.get_file_list()
distribution = dataset_loader.get_genre_distribution()

print(f"Total number of audio files: {len(file_list)}")
print(f"Number of genres: {len(dataset_loader.genres)}")
print(f"\nGenres: {dataset_loader.genres}")

df_dist = pd.DataFrame(list(distribution.items()), columns=['Genre', 'Count'])
print("\nGenre Distribution:")
print(df_dist)

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(df_dist['Genre'], df_dist['Count'], color=sns.color_palette("husl", len(df_dist)))
plt.xlabel('Genre', fontsize=12)
plt.ylabel('Number of Files', fontsize=12)
plt.title('Distribution of Audio Files per Genre', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.2 Audio Visualization

In [ ]:
if len(file_list) > 0:
    sample_files = {}
    for genre in dataset_loader.genres:
        for file_path, g, _ in file_list:
            if g == genre:
                sample_files[genre] = file_path
                break
    
    for genre, file_path in list(sample_files.items())[:3]:
        audio, sr = librosa.load(file_path, sr=config['audio']['sample_rate'], duration=30)
        
        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        
        librosa.display.waveshow(audio, sr=sr, ax=axes[0])
        axes[0].set_title(f'Waveform - {genre}', fontweight='bold')
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Amplitude')
        
        D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[1])
        axes[1].set_title(f'Spectrogram - {genre}', fontweight='bold')
        fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
        
        melspec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
        img2 = librosa.display.specshow(melspec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[2])
        axes[2].set_title(f'Mel-Spectrogram - {genre}', fontweight='bold')
        fig.colorbar(img2, ax=axes[2], format='%+2.0f dB')
        
        plt.tight_layout()
        plt.show()
else:
    print("No audio files found. Please add data to data/raw/")

## 3. Feature Extraction and Analysis

### 3.1 Extract Features from Sample Files

In [ ]:
if len(file_list) > 0:
    X_features = []
    y_labels = []
    
    max_samples = min(100, len(file_list))
    
    for idx, (file_path, genre, label) in enumerate(file_list[:max_samples]):
        try:
            segments, sr = preprocess_audio(
                file_path,
                sr=config['audio']['sample_rate'],
                segment_length=config['audio']['segment_length'],
                mono=config['audio']['mono']
            )
            
            for segment in segments[:1]:
                mfcc = extract_mfcc_features(
                    segment, sr,
                    n_mfcc=config['features']['mfcc']['n_mfcc'],
                    n_fft=config['audio']['n_fft'],
                    hop_length=config['audio']['hop_length'],
                    n_mels=config['features']['mfcc']['n_mels']
                )
                
                spectral = extract_all_spectral_features(
                    segment, sr,
                    n_fft=config['audio']['n_fft'],
                    hop_length=config['audio']['hop_length'],
                    n_chroma=config['features']['chroma']['n_chroma'],
                    n_bands=config['features']['spectral']['n_bands']
                )
                
                features = np.concatenate([mfcc, spectral])
                X_features.append(features)
                y_labels.append(label)
            
            if (idx + 1) % 20 == 0:
                print(f"Processed {idx + 1}/{max_samples} files")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
    
    X_features = np.array(X_features)
    y_labels = np.array(y_labels)
    
    print(f"\nFeature matrix shape: {X_features.shape}")
else:
    print("No audio files found.")

### 3.2 Feature Visualization with PCA

In [ ]:
if len(file_list) > 0 and len(X_features) > 0:
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_features)
    
    plt.figure(figsize=(12, 8))
    
    for label in np.unique(y_labels):
        mask = y_labels == label
        genre_name = dataset_loader.idx_to_genre[label]
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=genre_name, alpha=0.6, s=100)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
    plt.title('PCA Visualization of Audio Features', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Total variance explained by 2 components: {sum(pca.explained_variance_ratio_):.2%}")

### 3.3 t-SNE Visualization

In [ ]:
if len(file_list) > 0 and len(X_features) > 0:
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_features)-1))
    X_tsne = tsne.fit_transform(X_features)
    
    plt.figure(figsize=(12, 8))
    
    for label in np.unique(y_labels):
        mask = y_labels == label
        genre_name = dataset_loader.idx_to_genre[label]
        plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=genre_name, alpha=0.6, s=100)
    
    plt.xlabel('t-SNE 1', fontsize=12)
    plt.ylabel('t-SNE 2', fontsize=12)
    plt.title('t-SNE Visualization of Audio Features', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Model Results Comparison

### 4.1 Load Model Results

In [ ]:
models = ['knn', 'svm', 'rf', 'mlp', 'gmm', 'cnn']
results = {}

for model in models:
    results_path = f"../outputs/{model}_results.json"
    if Path(results_path).exists():
        with open(results_path, 'r') as f:
            results[model.upper()] = json.load(f)
        print(f"Loaded results for {model.upper()}")
    else:
        print(f"Results not found for {model.upper()}. Train the model first.")

print(f"\nLoaded {len(results)} model results")

### 4.2 Performance Metrics Comparison

In [ ]:
if results:
    metrics_df = pd.DataFrame({
        model: {
            'Accuracy': data['metrics']['accuracy'],
            'Precision': data['metrics']['precision_macro'],
            'Recall': data['metrics']['recall_macro'],
            'F1-Score': data['metrics']['f1_macro']
        }
        for model, data in results.items()
    }).T
    
    print("Model Performance Comparison:")
    print(metrics_df.round(4))
    
    metrics_df.plot(kind='bar', figsize=(14, 6))
    plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
    plt.ylabel('Score', fontsize=12)
    plt.xlabel('Model', fontsize=12)
    plt.ylim([0, 1.0])
    plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("No model results available.")

### 4.3 Best Model Identification

In [ ]:
if results:
    best_model = metrics_df['Accuracy'].idxmax()
    best_accuracy = metrics_df['Accuracy'].max()
    
    print(f"Best Model: {best_model}")
    print(f"Best Accuracy: {best_accuracy:.4f}")
    print(f"\nBest Model Performance:")
    print(metrics_df.loc[best_model].round(4))

### 4.4 Confusion Matrices

In [ ]:
if results and len(dataset_loader.genres) > 0:
    n_models = len(results)
    n_cols = 2
    n_rows = (n_models + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
    axes = axes.flatten() if n_models > 1 else [axes]
    
    for idx, (model_name, data) in enumerate(results.items()):
        cm = np.array(data['confusion_matrix'])
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=dataset_loader.genres,
            yticklabels=dataset_loader.genres,
            ax=axes[idx],
            cbar_kws={'label': 'Count'}
        )
        axes[idx].set_title(f'{model_name} Confusion Matrix', fontweight='bold')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    for idx in range(len(results), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

### 4.5 Training History (MLP and CNN)

In [ ]:
history_plots = []
for model in ['mlp', 'cnn']:
    plot_path = f"../outputs/plots/{model}_training_history.png"
    if Path(plot_path).exists():
        history_plots.append((model.upper(), plot_path))

if history_plots:
    from IPython.display import Image, display
    
    for model_name, plot_path in history_plots:
        print(f"\n{model_name} Training History:")
        display(Image(filename=plot_path))
else:
    print("No training history plots found.")

## 5. Conclusions and Summary

### 5.1 Summary Statistics

In [ ]:
if results:
    print("Summary of Model Performance:")
    print("=" * 80)
    
    summary_df = pd.DataFrame({
        'Model': list(results.keys()),
        'Accuracy': [data['metrics']['accuracy'] for data in results.values()],
        'F1-Score': [data['metrics']['f1_macro'] for data in results.values()],
        'Precision': [data['metrics']['precision_macro'] for data in results.values()],
        'Recall': [data['metrics']['recall_macro'] for data in results.values()]
    })
    
    summary_df = summary_df.sort_values('Accuracy', ascending=False)
    print(summary_df.to_string(index=False))
    
    print("\n" + "=" * 80)
    print(f"Average Accuracy: {summary_df['Accuracy'].mean():.4f}")
    print(f"Standard Deviation: {summary_df['Accuracy'].std():.4f}")
    print(f"Best Model: {summary_df.iloc[0]['Model']} ({summary_df.iloc[0]['Accuracy']:.4f})")
    print(f"Worst Model: {summary_df.iloc[-1]['Model']} ({summary_df.iloc[-1]['Accuracy']:.4f})")

### 5.2 Key Findings

This project successfully implemented and compared multiple approaches for music genre classification:

**Traditional Machine Learning Models:**
- KNN: Simple distance-based classification
- SVM: Powerful kernel-based method
- Random Forest: Ensemble method with good generalization
- MLP: Neural network with dense layers
- GMM: Probabilistic model using Gaussian mixtures

**Deep Learning Model:**
- CNN: Convolutional network processing mel-spectrograms

**Feature Extraction:**
- MFCCs capture timbral characteristics
- Spectral features provide frequency domain information
- Mel-spectrograms offer time-frequency representations

**Observations:**
- Different models show varying performance based on the dataset
- Feature engineering is crucial for traditional ML models
- CNN can learn hierarchical features from spectrograms
- Ensemble methods generally show robust performance

**Future Improvements:**
- Data augmentation techniques
- Transfer learning from pre-trained audio models
- Ensemble of multiple models
- More sophisticated architectures (ResNet, Transformer)
- Hyperparameter optimization

## End of Report